In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Start rapid cu Sampler

Sarcina principală a Sampler este eșantionarea registrului de ieșire din execuția unuia sau mai multor circuite cuantice. [Circuitele dinamice](/guides/execute-dynamic-circuits) și circuitele parametrizate sunt acceptate ca intrare (dacă sunt trimise circuite parametrizate, valorile parametrilor trebuie furnizate de asemenea). Sampler suportă, de asemenea, decuplare dinamică și twirling integrate pentru [suprimarea erorilor](/guides/error-mitigation-and-suppression-techniques).

Pașii din acest subiect descriu cum să configurezi Sampler, să explorezi opțiunile pe care le poți folosi pentru a-l configura și să îl invoci într-un program.


<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unor versiuni mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

## Pași pentru utilizarea primitivei Sampler
### 1. Inițializarea contului
Deoarece Qiskit Runtime este un serviciu gestionat, trebuie mai întâi să îți inițializezi contul. Poți selecta apoi QPU-ul pe care vrei să îl folosești pentru a calcula valoarea de așteptare.

Urmează pașii din subiectul [Configurarea contului IBM Cloud](/guides/cloud-setup) dacă nu ai deja un cont configurat.

:::note[Gate-uri fracționale]

Pentru a utiliza [gate-urile fracționale](/guides/fractional-gates) nou suportate, setează `use_fractional_gates=True` atunci când soliciți un backend dintr-o instanță `QiskitRuntimeService`. De exemplu:

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

Aceasta este o caracteristică experimentală și s-ar putea schimba în viitor.

:::

In [2]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(127, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

### 2. Crearea unui circuit
Ai nevoie de cel puțin un circuit ca intrare pentru primitiva Sampler.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 3036), ('sx', 1769), ('cz', 378), ('measure', 127), ('barrier', 1)])


Circuitul și observabilul trebuie transformate pentru a utiliza doar instrucțiunile suportate de QPU (denumite circuite de *arhitectură set de instrucțiuni (ISA)*). Folosește transpilerul pentru aceasta.

In [4]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

### 4. Invoke Sampler and get results

Next, invoke the `run()` method to generate the output. The circuit and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d82863mgbeec73alf9sg


>>> Job Status: QUEUED


In [ ]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(
    f"First ten results for the 'meas' output register: "
    f"{pub_result.data.meas.get_bitstrings()[:10]}"
)

First ten results for the 'meas' output register: ['1100110011001011111111111010000010001010100100011000001011001101000110011000110100100100101010111001110100100000000011111100000', '0101001001010000100111000110110001001101010110110000110111101110001100000001000001111111101110000000010011111100100110001101000', '0111111110011011000011110111010111101100110010001010010001100000000100000000001010101010111010110000001100100001010110000101000', '0000110011001100110011101100000111011001110100001100001100110111010100101010001010000011000111001010101111110110100110001010000', '0011110011100001100110111001000011011111011110111100000110001000111011101101000110011011101011001110110000010010001100100011001', '1010001000010101011100101010101001101000100010011011100110010111010001110111110010100010111010011010110011001101100110010000010', '0001110010001011001100010000000001001101001110101100110011101111100100100110110010101000011010101000101011101011010100000101010', '1110100100001100110010000010011

### 4. Invocarea Sampler și obținerea rezultatelor
Apoi, invocă metoda `run()` pentru a genera ieșirea. Circuitul și seturile opționale de valori de parametri sunt introduse ca tupluri *primitive unified bloc* (PUB).